# Week 3 · Lecture 5 — Gaussian Elimination & LU Decomposition

> **Linear Algebra for ML** · a first-principles course
>
> *derive → implement → visualize → verify → connect to ML*

How does a computer actually solve $A\mathbf{x} = \mathbf{b}$? Not by inverting $A$ — that is slow
and numerically dangerous. It uses **Gaussian elimination**: systematically zero out entries below
the diagonal until the system is triangular and trivially solvable by back-substitution.

Bottling that process into a reusable factorization gives the **LU decomposition** $A = LU$, the
workhorse behind `scipy.linalg.solve`. We build both by hand, then confront the reason **pivoting**
exists: without it, floating-point errors explode.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu as scipy_lu
from utils.linalg_viz import check

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 110

## 1. Back-substitution: solving a triangular system

If a system is already **upper-triangular**, solving it is easy: the last equation has one unknown,
and we work upward, substituting as we go. This is the payoff step of elimination, so we build it
first.

In [2]:
def back_substitution(U, b):
    U, b = np.asarray(U, float), np.asarray(b, float)
    n = len(b)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

U = np.array([[2.0, 1.0, -1.0],
              [0.0, 3.0,  2.0],
              [0.0, 0.0,  4.0]])
b = np.array([5.0, 4.0, 8.0])
x = back_substitution(U, b)
check("back-substitution", U @ x, b)
print("solution x =", x)

[PASS] back-substitution                max|Δ| = 0.00e+00
solution x = [3.5 0.  2. ]


## 2. Gaussian elimination → LU

Elimination repeatedly subtracts a multiple of one row from those below it to create zeros under
the pivot. The crucial insight: **the multipliers we use are exactly the entries of a lower-triangular
matrix $L$**, and what remains is the upper-triangular $U$. So elimination *is* the factorization

$$ A = LU, \qquad L = \text{unit lower-triangular (the multipliers)}, \quad U = \text{upper-triangular}. $$

Once we have $L$ and $U$, solving $A\mathbf{x}=\mathbf{b}$ becomes two cheap triangular solves:
$L\mathbf{y}=\mathbf{b}$ then $U\mathbf{x}=\mathbf{y}$. Let's implement the unpivoted version.

In [3]:
def lu_decompose(A):
    A = np.asarray(A, float).copy()
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]      # the multiplier
            L[i, k] = m                # store it in L
            U[i] = U[i] - m * U[k]     # eliminate below the pivot
    return L, U

A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])
L, U = lu_decompose(A)
print("L =\n", L, "\n\nU =\n", U)
check("A == L U", L @ U, A)

L =
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]] 

U =
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
[PASS] A == L U                         max|Δ| = 0.00e+00


True

In [4]:
def forward_substitution(L, b):
    L, b = np.asarray(L, float), np.asarray(b, float)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

def lu_solve(A, b):
    L, U = lu_decompose(A)
    y = forward_substitution(L, b)     # L y = b
    return back_substitution(U, y)     # U x = y

b = np.array([4.0, 10.0, 24.0])
x = lu_solve(A, b)
check("lu_solve vs np.linalg.solve", x, np.linalg.solve(A, b))
print("solution x =", x)

[PASS] lu_solve vs np.linalg.solve      max|Δ| = 7.77e-16
solution x = [1. 1. 1.]


## 3. Why pivoting matters: instability is real

The unpivoted algorithm divides by the pivot $U_{kk}$. If a pivot is tiny, the multipliers explode
and floating-point error swamps the answer. The fix — **partial pivoting** — swaps the largest
available entry into the pivot position at each step. This is why production code returns $PA = LU$
with a permutation $P$.

The example below has a catastrophically small pivot; watch the unpivoted solution drift while the
pivoted library result stays correct.

In [5]:
eps = 1e-15
A_bad = np.array([[eps, 1.0],
                  [1.0, 1.0]])
b = np.array([1.0, 2.0])

x_naive = lu_solve(A_bad, b)             # no pivoting
x_ref   = np.linalg.solve(A_bad, b)      # pivoted, reliable

print("naive (no pivot) :", x_naive)
print("reference (pivot):", x_ref)
print("the true answer is ≈ [1, 1]; the naive pivot-free result is wrong.")

# scipy's PA = LU shows the permutation that fixes it
P, L, U = scipy_lu(A_bad)
print("\npermutation P (rows were swapped):\n", P)

naive (no pivot) : [0.9992 1.    ]
reference (pivot): [1. 1.]
the true answer is ≈ [1, 1]; the naive pivot-free result is wrong.

permutation P (rows were swapped):
 [[0. 1.]
 [1. 0.]]


## 4. Conditioning — how sensitive is the answer?

Even with perfect pivoting, some matrices are intrinsically fragile: a tiny change in $\mathbf{b}$
produces a huge change in $\mathbf{x}$. The **condition number** $\kappa(A)$ quantifies this. A
large $\kappa$ means "ill-conditioned" — the system amplifies errors. We'll see this exact quantity
return in Week 6 as the ratio of largest to smallest singular value.

In [6]:
well = np.array([[2.0, 0.0], [0.0, 2.0]])
ill  = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-6]])

for name, M in [("well-conditioned", well), ("ill-conditioned", ill)]:
    print(f"{name:<18} κ(A) = {np.linalg.cond(M):.3e}")
print("\nLarge κ ⇒ the solution is hypersensitive to noise in the data.")

well-conditioned   κ(A) = 1.000e+00
ill-conditioned    κ(A) = 4.000e+06

Large κ ⇒ the solution is hypersensitive to noise in the data.


## 5. Where this shows up in ML

- **The normal equations** $X^\top X \beta = X^\top y$ (next lecture, linear regression) are solved
  with exactly this triangular machinery.
- **Conditioning explains training pain.** Ill-conditioned $X^\top X$ — caused by correlated
  features or bad scaling — makes regression unstable and gradient descent slow; it motivates
  feature standardization and ridge regularization.
- **LU/Cholesky factorizations** underlie Gaussian processes, Kalman filters, and second-order
  optimizers that need to solve linear systems repeatedly.

In [7]:
# timing intuition: factor once, solve many right-hand sides cheaply
A = np.random.default_rng(0).normal(size=(200, 200))
A = A @ A.T + 200 * np.eye(200)          # make it nicely conditioned (SPD)
L, U = lu_decompose(A)

B = np.random.default_rng(1).normal(size=(200, 5))   # 5 different targets
X = np.column_stack([back_substitution(U, forward_substitution(L, b)) for b in B.T])
check("multi-RHS solve", A @ X, B, atol=1e-6)
print("One factorization reused across 5 right-hand sides — the LU payoff.")

[PASS] multi-RHS solve                  max|Δ| = 5.11e-15
One factorization reused across 5 right-hand sides — the LU payoff.


## Exercises

1. **Count the operations.** Roughly how many multiply–adds does eliminating an $n\times n$ matrix take? (Hint: it scales like $n^3$.)
2. **Pivot by hand.** Implement partial pivoting inside `lu_decompose` (track a permutation) and re-solve the unstable example from §3.
3. **Determinant from U.** After elimination, $\det A = \pm\prod_i U_{ii}$. Verify this on a random matrix (sign comes from the number of row swaps).

In [8]:
# === Solutions (try first!) ===

# 1. ~ (2/3) n^3 multiply-adds → "Gaussian elimination is O(n^3)".

# 3. determinant as product of pivots (no pivoting → sign +1 here)
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L, U = lu_decompose(A)
det_from_U = np.prod(np.diag(U))
check("det from pivots", det_from_U, np.linalg.det(A), atol=1e-6)
print("product of U's diagonal =", round(det_from_U, 6))

[PASS] det from pivots                  max|Δ| = 8.88e-16
product of U's diagonal = 4.0


## Recap & what's next

Elimination turns any system triangular; the multipliers form $L$ and the result is $U$, giving
$A = LU$. Pivoting keeps it numerically safe, and the condition number tells us how much to trust
the answer.

**Next — `06_least_squares_normal_equations.ipynb`:** real data has more equations than unknowns,
so there is *no* exact solution. We'll project the target onto the column space — connecting
Lecture 2's projection to Lecture 4's column space — and out pops **linear regression**.

---
*Linear Algebra for ML · Week 3 · Lecture 5*